In [ ]:
import requests
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN

url = "https://data.geo.admin.ch/ch.bfe.ladestellen-elektromobilitaet/data/ch.bfe.ladestellen-elektromobilitaet.json"
data = requests.get(url).json()

# --------------------------------------------------
# 1. Flatten all operators into one EVSE-level table
# --------------------------------------------------
all_rows = []

for op in data["EVSEData"]:
    operator_name = op.get("OperatorName", "Unknown")
    records = op.get("EVSEDataRecord", [])

    if not records:
        continue

    df_op = pd.json_normalize(records)
    df_op["OperatorName"] = operator_name
    all_rows.append(df_op)

df_all_records = pd.concat(all_rows, ignore_index=True)

print("Total EVSE records:", len(df_all_records))
print("Operators:", df_all_records["OperatorName"].nunique())
print(df_all_records.head())


# --------------------------------------------------
# 2. Extract coordinates from GeoCoordinates.Google
# --------------------------------------------------
# expected format: "46.971079 8.459159"
coords_split = df_all_records["GeoCoordinates.Google"].astype(str).str.split(" ", expand=True)

df_all_records["lat"] = pd.to_numeric(coords_split[0], errors="coerce")
df_all_records["lon"] = pd.to_numeric(coords_split[1], errors="coerce")

df_all_records = df_all_records.dropna(subset=["lat", "lon"]).reset_index(drop=True)

print("Records with valid coordinates:", len(df_all_records))


# --------------------------------------------------
# 3. Simple aggregation by exact address + coordinates
# --------------------------------------------------
group_cols = [
    "OperatorName",
    "Address.City",
    "Address.PostalCode",
    "Address.Street",
    "Address.HouseNum",
    "lat",
    "lon"
]

df_all_sites = (
    df_all_records
    .groupby(group_cols, dropna=False)
    .agg(
        charger_count=("EvseID", "count")
    )
    .reset_index()
    .sort_values(["OperatorName", "charger_count"], ascending=[True, False])
)

print("Aggregated sites:", len(df_all_sites))
print(df_all_sites.head(20))


# --------------------------------------------------
# 4. Operator summary
# --------------------------------------------------
df_operator_summary = (
    df_all_sites
    .groupby("OperatorName")
    .agg(
        site_count=("OperatorName", "count"),
        total_chargers=("charger_count", "sum"),
        max_site_size=("charger_count", "max"),
        avg_site_size=("charger_count", "mean")
    )
    .reset_index()
    .sort_values("total_chargers", ascending=False)
)

print(df_operator_summary.head(50))


# --------------------------------------------------
# 5. Optional: DBSCAN clustering per operator
#    useful if same site has slightly different coords
# --------------------------------------------------
def cluster_operator_sites(df_operator, eps_meters=30):
    df_operator = df_operator.copy()

    coords_rad = np.radians(df_operator[["lat", "lon"]].to_numpy())
    earth_radius_m = 6371000
    eps = eps_meters / earth_radius_m

    db = DBSCAN(eps=eps, min_samples=1, metric="haversine")
    df_operator["cluster_id"] = db.fit_predict(coords_rad)

    df_clustered = (
        df_operator.groupby("cluster_id", dropna=False)
        .agg(
            OperatorName=("OperatorName", "first"),
            charger_count=("EvseID", "count"),
            site_lat=("lat", "mean"),
            site_lon=("lon", "mean"),
            city=("Address.City", "first"),
            postal_code=("Address.PostalCode", "first"),
            street=("Address.Street", "first"),
            house_num=("Address.HouseNum", "first")
        )
        .reset_index()
    )

    return df_operator, df_clustered


clustered_records = []
clustered_sites = []

for operator_name, df_op in df_all_records.groupby("OperatorName"):
    df_op_clustered, df_op_sites = cluster_operator_sites(df_op, eps_meters=30)
    clustered_records.append(df_op_clustered)
    clustered_sites.append(df_op_sites)

df_all_records_clustered = pd.concat(clustered_records, ignore_index=True)
df_all_sites_clustered = pd.concat(clustered_sites, ignore_index=True)

print("Clustered sites total:", len(df_all_sites_clustered))
print(df_all_sites_clustered.head(20))


# --------------------------------------------------
# 6. Cluster-based operator summary
# --------------------------------------------------
df_operator_summary_clustered = (
    df_all_sites_clustered
    .groupby("OperatorName")
    .agg(
        site_count=("OperatorName", "count"),
        total_chargers=("charger_count", "sum"),
        max_site_size=("charger_count", "max"),
        avg_site_size=("charger_count", "mean")
    )
    .reset_index()
    .sort_values("total_chargers", ascending=False)
)

print(df_operator_summary_clustered.head(50))


# --------------------------------------------------
# 7. Save to CSV
# --------------------------------------------------
df_all_records.to_csv("all_operators_evse_records.csv", index=False)
df_all_sites.to_csv("all_operators_sites_exact.csv", index=False)
df_all_sites_clustered.to_csv("all_operators_sites_clustered.csv", index=False)
df_operator_summary.to_csv("operator_summary_exact.csv", index=False)
df_operator_summary_clustered.to_csv("operator_summary_clustered.csv", index=False)

In [ ]:
import requests
import pandas as pd

url = "https://data.geo.admin.ch/ch.bfe.ladestellen-elektromobilitaet/data/ch.bfe.ladestellen-elektromobilitaet.json"
data = requests.get(url).json()

all_rows = []

for op in data["EVSEData"]:
    operator_name = op.get("OperatorName", "Unknown")
    records = op.get("EVSEDataRecord", [])
    if not records:
        continue

    df_op = pd.json_normalize(records)
    df_op["OperatorName"] = operator_name
    all_rows.append(df_op)

df = pd.concat(all_rows, ignore_index=True)

coords = df["GeoCoordinates.Google"].astype(str).str.split(" ", expand=True)
df["lat"] = pd.to_numeric(coords[0], errors="coerce")
df["lon"] = pd.to_numeric(coords[1], errors="coerce")
df = df.dropna(subset=["lat", "lon"]).reset_index(drop=True)

df_sites = (
    df.groupby([
        "OperatorName",
        "Address.City",
        "Address.PostalCode",
        "Address.Street",
        "Address.HouseNum",
        "lat",
        "lon"
    ], dropna=False)
    .size()
    .reset_index(name="charger_count")
    .sort_values(["OperatorName", "charger_count"], ascending=[True, False])
)

print(df_sites.head(50))

In [ ]:
df_summary = (
    df_sites.groupby("OperatorName")
    .agg(
        site_count=("OperatorName", "count"),
        total_chargers=("charger_count", "sum"),
        avg_chargers_per_site=("charger_count", "mean"),
        max_chargers_one_site=("charger_count", "max")
    )
    .reset_index()
    .sort_values("total_chargers", ascending=False)
)

print(df_summary)

In [ ]:
top_ops = (
    df_all_sites_clustered.groupby("OperatorName")["charger_count"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .index
)

df_clusters = df_all_sites_clustered[
    df_all_sites_clustered["OperatorName"].isin(top_ops)
].copy()

In [ ]:
import folium
import numpy as np
import pandas as pd

from folium.plugins import HeatMap
from scipy.spatial import Voronoi
from shapely.geometry import Polygon, box

CENTER = [46.8182, 8.2275]
ZOOM_START = 8

# ---------------------------------------------------------
# INPUT
# ---------------------------------------------------------
# expected columns:
# site_lat, site_lon, charger_count, city
#
# example:
# df_clusters = df_all_sites_clustered.copy()

df_map = df_clusters.copy()

# keep only valid coordinates
df_map = df_map.dropna(subset=["site_lat", "site_lon"]).copy()
df_map["site_lat"] = pd.to_numeric(df_map["site_lat"], errors="coerce")
df_map["site_lon"] = pd.to_numeric(df_map["site_lon"], errors="coerce")
df_map["charger_count"] = pd.to_numeric(df_map["charger_count"], errors="coerce").fillna(1)
df_map = df_map.dropna(subset=["site_lat", "site_lon"]).reset_index(drop=True)

# optional: merge exact duplicate site coords
df_map = (
    df_map.groupby(["site_lat", "site_lon"], dropna=False)
    .agg(
        charger_count=("charger_count", "sum"),
        city=("city", "first") if "city" in df_map.columns else ("site_lat", "size")
    )
    .reset_index()
)

print("Sites used for map:", len(df_map))


# ---------------------------------------------------------
# helper: finite voronoi regions
# adapted to produce closed polygons
# ---------------------------------------------------------
def voronoi_finite_polygons_2d(vor, radius=None):
    if vor.points.shape[1] != 2:
        raise ValueError("Requires 2D input")

    new_regions = []
    new_vertices = vor.vertices.tolist()

    center = vor.points.mean(axis=0)
    if radius is None:
        radius = vor.points.ptp().max() * 2

    all_ridges = {}

    for (p1, p2), (v1, v2) in zip(vor.ridge_points, vor.ridge_vertices):
        all_ridges.setdefault(p1, []).append((p2, v1, v2))
        all_ridges.setdefault(p2, []).append((p1, v1, v2))

    for p1, region_idx in enumerate(vor.point_region):
        vertices = vor.regions[region_idx]

        if all(v >= 0 for v in vertices):
            new_regions.append(vertices)
            continue

        ridges = all_ridges[p1]
        new_region = [v for v in vertices if v >= 0]

        for p2, v1, v2 in ridges:
            if v1 < 0 or v2 < 0:
                v = v1 if v1 >= 0 else v2
                t = vor.points[p2] - vor.points[p1]
                t = t / np.linalg.norm(t)
                n = np.array([-t[1], t[0]])

                midpoint = vor.points[[p1, p2]].mean(axis=0)
                direction = np.sign(np.dot(midpoint - center, n)) * n
                far_point = vor.vertices[v] + direction * radius

                new_region.append(len(new_vertices))
                new_vertices.append(far_point.tolist())

        vs = np.asarray([new_vertices[v] for v in new_region])
        c = vs.mean(axis=0)
        angles = np.arctan2(vs[:, 1] - c[1], vs[:, 0] - c[0])
        new_region = [v for _, v in sorted(zip(angles, new_region))]
        new_regions.append(new_region)

    return new_regions, np.asarray(new_vertices)


# ---------------------------------------------------------
# helper: build Voronoi polygons clipped to Switzerland-ish bbox
# coordinates are lon/lat
# ---------------------------------------------------------
def build_clipped_voronoi(df_points, bbox=(5.8, 45.7, 10.7, 47.95)):
    """
    bbox = (min_lon, min_lat, max_lon, max_lat)
    """
    if len(df_points) < 3:
        return []

    points = df_points[["site_lon", "site_lat"]].to_numpy()
    vor = Voronoi(points)
    regions, vertices = voronoi_finite_polygons_2d(vor, radius=10)

    clip_box = box(*bbox)
    polygons = []

    for i, region in enumerate(regions):
        poly_coords = vertices[region]
        poly = Polygon(poly_coords)

        if not poly.is_valid:
            poly = poly.buffer(0)

        if poly.is_empty:
            continue

        clipped = poly.intersection(clip_box)

        if clipped.is_empty:
            continue

        polygons.append((i, clipped))

    return polygons


# ---------------------------------------------------------
# map
# ---------------------------------------------------------
m = folium.Map(location=CENTER, zoom_start=ZOOM_START, tiles=None)

folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_nolabels/{z}/{x}/{y}{r}.png",
    attr="&copy; OpenStreetMap contributors &copy; CARTO",
    overlay=False,
    control=False
).add_to(m)

folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_only_labels/{z}/{x}/{y}{r}.png",
    attr="&copy; OpenStreetMap contributors &copy; CARTO",
    overlay=True,
    control=False
).add_to(m)


# ---------------------------------------------------------
# weighted heatmap
# ---------------------------------------------------------
heat_data = [
    [row["site_lat"], row["site_lon"], row["charger_count"]]
    for _, row in df_map.iterrows()
]

HeatMap(
    heat_data,
    radius=25,
    blur=18,
    min_opacity=0.25,
    max_zoom=12
).add_to(m)


# ---------------------------------------------------------
# Voronoi polygons
# ---------------------------------------------------------
vor_polys = build_clipped_voronoi(df_map)

for idx, geom in vor_polys:
    # handle polygon / multipolygon
    geoms = [geom] if geom.geom_type == "Polygon" else list(geom.geoms)

    for g in geoms:
        coords = [[lat, lon] for lon, lat in np.array(g.exterior.coords)]

        row = df_map.iloc[idx]
        tooltip = f"{int(row['charger_count'])} chargers"
        if "city" in df_map.columns:
            tooltip += f" | {row['city']}"

        folium.Polygon(
            locations=coords,
            color="red",
            weight=1,
            fill=False,
            opacity=0.45,
            tooltip=tooltip
        ).add_to(m)


# ---------------------------------------------------------
# site markers
# ---------------------------------------------------------
for _, row in df_map.iterrows():
    tooltip = f"{int(row['charger_count'])} chargers"
    if "city" in df_map.columns:
        tooltip += f" | {row['city']}"

    folium.CircleMarker(
        location=[row["site_lat"], row["site_lon"]],
        radius=min(4 + row["charger_count"] * 0.35, 16),
        color="red",
        weight=2,
        fill=True,
        fill_color="red",
        fill_opacity=0.85,
        tooltip=tooltip
    ).add_to(m)

m

In [ ]:
import warnings
import numpy as np
from scipy.spatial import Voronoi
from shapely.geometry import Polygon, box, MultiPolygon
from shapely.errors import GEOSException

# optional: silence only this noisy shapely runtime warning
warnings.filterwarnings(
    "ignore",
    message="invalid value encountered in intersection",
    category=RuntimeWarning
)

def voronoi_finite_polygons_2d(vor, radius=None):
    if vor.points.shape[1] != 2:
        raise ValueError("Requires 2D input")

    new_regions = []
    new_vertices = vor.vertices.tolist()

    center = vor.points.mean(axis=0)
    if radius is None:
        radius = np.ptp(vor.points, axis=0).max() * 2

    all_ridges = {}

    for (p1, p2), (v1, v2) in zip(vor.ridge_points, vor.ridge_vertices):
        all_ridges.setdefault(p1, []).append((p2, v1, v2))
        all_ridges.setdefault(p2, []).append((p1, v1, v2))

    for p1, region_idx in enumerate(vor.point_region):
        vertices = vor.regions[region_idx]

        if not vertices:
            continue

        if all(v >= 0 for v in vertices):
            new_regions.append(vertices)
            continue

        ridges = all_ridges.get(p1, [])
        new_region = [v for v in vertices if v >= 0]

        for p2, v1, v2 in ridges:
            if v1 < 0 or v2 < 0:
                v = v1 if v1 >= 0 else v2

                if v < 0:
                    continue

                t = vor.points[p2] - vor.points[p1]
                norm = np.linalg.norm(t)
                if norm == 0:
                    continue

                t = t / norm
                n = np.array([-t[1], t[0]])

                midpoint = vor.points[[p1, p2]].mean(axis=0)
                direction = np.sign(np.dot(midpoint - center, n)) * n
                far_point = vor.vertices[v] + direction * radius

                new_region.append(len(new_vertices))
                new_vertices.append(far_point.tolist())

        if len(new_region) == 0:
            continue

        vs = np.asarray([new_vertices[v] for v in new_region])

        if len(vs) < 3:
            continue

        c = vs.mean(axis=0)
        angles = np.arctan2(vs[:, 1] - c[1], vs[:, 0] - c[0])
        new_region = [v for _, v in sorted(zip(angles, new_region))]
        new_regions.append(new_region)

    return new_regions, np.asarray(new_vertices)


def safe_clipped_voronoi(df_points, bbox=(5.8, 45.7, 10.7, 47.95), radius=10):
    if len(df_points) < 3:
        return []

    points = df_points[["site_lon", "site_lat"]].to_numpy()
    vor = Voronoi(points)
    regions, vertices = voronoi_finite_polygons_2d(vor, radius=radius)

    clip_box = box(*bbox)
    polygons = []

    for i, region in enumerate(regions):
        try:
            poly_coords = vertices[region]

            if len(poly_coords) < 3:
                continue

            # remove NaN / inf
            if not np.isfinite(poly_coords).all():
                continue

            poly = Polygon(poly_coords)

            if poly.is_empty:
                continue

            if not poly.is_valid:
                poly = poly.buffer(0)

            if poly.is_empty or not poly.is_valid:
                continue

            # extra sanity check
            minx, miny, maxx, maxy = poly.bounds
            if not np.isfinite([minx, miny, maxx, maxy]).all():
                continue

            clipped = poly.intersection(clip_box)

            if clipped.is_empty:
                continue

            if not clipped.is_valid:
                clipped = clipped.buffer(0)

            if clipped.is_empty:
                continue

            polygons.append((i, clipped))

        except (GEOSException, ValueError, FloatingPointError):
            continue

    return polygons

In [ ]:
import folium
import pandas as pd
from folium.plugins import HeatMap

CENTER = [47.38922397548196, 8.517438420931438]
ZOOM_START = 10

# ---------------------------------------------------------
# input
# ---------------------------------------------------------
df_map = df_all_sites_clustered.copy()

# clean
df_map = df_map.dropna(subset=["site_lat", "site_lon", "OperatorName"]).copy()
df_map["site_lat"] = pd.to_numeric(df_map["site_lat"], errors="coerce")
df_map["site_lon"] = pd.to_numeric(df_map["site_lon"], errors="coerce")
df_map["charger_count"] = pd.to_numeric(df_map["charger_count"], errors="coerce").fillna(1)
df_map = df_map.dropna(subset=["site_lat", "site_lon"]).reset_index(drop=True)

print("Sites used:", len(df_map))
print("Operators:", df_map["OperatorName"].nunique())

# ---------------------------------------------------------
# color palette
# ---------------------------------------------------------
palette = [
    "red", "blue", "green", "purple", "orange", "darkred", "lightred",
    "beige", "darkblue", "darkgreen", "cadetblue", "darkpurple",
    "white", "pink", "lightblue", "lightgreen", "gray", "black", "lightgray"
]

operators = sorted(df_map["OperatorName"].dropna().unique())
color_map = {op: palette[i % len(palette)] for i, op in enumerate(operators)}

print(color_map)

# ---------------------------------------------------------
# map
# ---------------------------------------------------------
m = folium.Map(location=CENTER, zoom_start=ZOOM_START, tiles=None)

folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_nolabels/{z}/{x}/{y}{r}.png",
    attr="&copy; OpenStreetMap contributors &copy; CARTO",
    overlay=False,
    control=False
).add_to(m)

folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_only_labels/{z}/{x}/{y}{r}.png",
    attr="&copy; OpenStreetMap contributors &copy; CARTO",
    overlay=True,
    control=False
).add_to(m)

# ---------------------------------------------------------
# optional global heatmap
# ---------------------------------------------------------
heat_data = [
    [row["site_lat"], row["site_lon"], row["charger_count"]]
    for _, row in df_map.iterrows()
]

HeatMap(
    heat_data,
    radius=25,
    blur=18,
    min_opacity=0.20,
    max_zoom=12,
    name="Heatmap"
).add_to(m)

# ---------------------------------------------------------
# one layer per operator
# ---------------------------------------------------------
for op in operators:
    fg = folium.FeatureGroup(name=op, show=False)
    df_op = df_map[df_map["OperatorName"] == op]

    for _, row in df_op.iterrows():
        color = color_map[op]

        city = row["city"] if "city" in row and pd.notna(row["city"]) else ""
        street = row["street"] if "street" in row and pd.notna(row["street"]) else ""
        house_num = row["house_num"] if "house_num" in row and pd.notna(row["house_num"]) else ""

        tooltip = f"{op} | {int(row['charger_count'])} chargers"
        if city:
            tooltip += f" | {city}"

        popup_html = f"""
        <b>Operator:</b> {op}<br>
        <b>Chargers:</b> {int(row['charger_count'])}<br>
        <b>City:</b> {city}<br>
        <b>Street:</b> {street} {house_num}<br>
        <b>Lat/Lon:</b> {row['site_lat']:.6f}, {row['site_lon']:.6f}
        """

        folium.CircleMarker(
            location=[row["site_lat"], row["site_lon"]],
            radius=min(4 + row["charger_count"] * 0.35, 16),
            color=color,
            weight=2,
            fill=True,
            fill_color=color,
            fill_opacity=0.85,
            tooltip=tooltip,
            popup=folium.Popup(popup_html, max_width=300)
        ).add_to(fg)

    fg.add_to(m)

# ---------------------------------------------------------
# legend
# ---------------------------------------------------------
legend_items = ""
for op in operators:
    color = color_map[op]
    legend_items += f"""
    <div style="margin-bottom:4px;">
        <span style="
            display:inline-block;
            width:12px;
            height:12px;
            background:{color};
            border:1px solid #333;
            margin-right:6px;
            vertical-align:middle;
        "></span>
        <span style="font-size:12px;">{op}</span>
    </div>
    """

legend_html = f"""
<div style="
    position: fixed;
    bottom: 40px;
    left: 40px;
    width: 240px;
    max-height: 350px;
    overflow-y: auto;
    z-index: 9999;
    background: white;
    border: 2px solid #888;
    border-radius: 6px;
    padding: 10px;
    font-size: 12px;
    box-shadow: 0 0 8px rgba(0,0,0,0.25);
">
    <div style="font-weight: bold; margin-bottom: 8px;">Operators</div>
    {legend_items}
</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))

# ---------------------------------------------------------
# layer control
# ---------------------------------------------------------
folium.LayerControl(collapsed=False).add_to(m)

m

In [ ]:
import folium
import numpy as np
import pandas as pd
from folium.plugins import HeatMap
from scipy.spatial import Voronoi

CENTER = [47.38922397548196, 8.517438420931438]
ZOOM_START = 10

df_map = df_all_sites_clustered.copy()
df_map = df_map.dropna(subset=["site_lat", "site_lon", "OperatorName"]).copy()
df_map["site_lat"] = pd.to_numeric(df_map["site_lat"], errors="coerce")
df_map["site_lon"] = pd.to_numeric(df_map["site_lon"], errors="coerce")
df_map["charger_count"] = pd.to_numeric(df_map["charger_count"], errors="coerce").fillna(1)
df_map = df_map.dropna(subset=["site_lat", "site_lon"]).reset_index(drop=True)

group_cols = ["OperatorName", "site_lat", "site_lon"]
agg_dict = {"charger_count": "sum"}

if "city" in df_map.columns:
    agg_dict["city"] = "first"
if "street" in df_map.columns:
    agg_dict["street"] = "first"
if "house_num" in df_map.columns:
    agg_dict["house_num"] = "first"
if "postal_code" in df_map.columns:
    agg_dict["postal_code"] = "first"

df_map = df_map.groupby(group_cols, dropna=False).agg(agg_dict).reset_index()

palette = [
    "red", "blue", "green", "purple", "orange", "darkred", "lightred",
    "beige", "darkblue", "darkgreen", "cadetblue", "darkpurple",
    "pink", "lightblue", "lightgreen", "gray", "black", "lightgray"
]

operators = sorted(df_map["OperatorName"].dropna().unique())
operator_colors = {op: palette[i % len(palette)] for i, op in enumerate(operators)}

station_counts = df_map.groupby("OperatorName").size().to_dict()
total_stations = len(df_map)


def voronoi_finite_polygons_2d(vor, radius=None):
    if vor.points.shape[1] != 2:
        raise ValueError("Requires 2D input")

    new_regions = []
    new_vertices = vor.vertices.tolist()
    center = vor.points.mean(axis=0)

    if radius is None:
        radius = np.ptp(vor.points, axis=0).max() * 2

    all_ridges = {}
    for (p1, p2), (v1, v2) in zip(vor.ridge_points, vor.ridge_vertices):
        all_ridges.setdefault(p1, []).append((p2, v1, v2))
        all_ridges.setdefault(p2, []).append((p1, v1, v2))

    for p1, region_idx in enumerate(vor.point_region):
        vertices = vor.regions[region_idx]

        if not vertices:
            continue

        if all(v >= 0 for v in vertices):
            new_regions.append((p1, vertices))
            continue

        ridges = all_ridges.get(p1, [])
        new_region = [v for v in vertices if v >= 0]

        for p2, v1, v2 in ridges:
            if v1 < 0 or v2 < 0:
                v = v1 if v1 >= 0 else v2
                if v < 0:
                    continue

                t = vor.points[p2] - vor.points[p1]
                norm = np.linalg.norm(t)
                if norm == 0:
                    continue

                t = t / norm
                n = np.array([-t[1], t[0]])

                midpoint = vor.points[[p1, p2]].mean(axis=0)
                direction = np.sign(np.dot(midpoint - center, n)) * n
                far_point = vor.vertices[v] + direction * radius

                new_region.append(len(new_vertices))
                new_vertices.append(far_point.tolist())

        if len(new_region) < 3:
            continue

        vs = np.asarray([new_vertices[v] for v in new_region])
        if not np.isfinite(vs).all():
            continue

        c = vs.mean(axis=0)
        angles = np.arctan2(vs[:, 1] - c[1], vs[:, 0] - c[0])
        new_region = [v for _, v in sorted(zip(angles, new_region))]
        new_regions.append((p1, new_region))

    return new_regions, np.asarray(new_vertices)


def simple_voronoi_regions(df_points, radius=5, bbox=(5.8, 45.7, 10.7, 47.95)):
    if len(df_points) < 3:
        return []

    points = df_points[["site_lon", "site_lat"]].to_numpy()
    vor = Voronoi(points)
    regions, vertices = voronoi_finite_polygons_2d(vor, radius=radius)

    out = []
    min_lon, min_lat, max_lon, max_lat = bbox

    for point_idx, region in regions:
        poly_coords = vertices[region]

        if len(poly_coords) < 3:
            continue
        if not np.isfinite(poly_coords).all():
            continue

        centroid = poly_coords.mean(axis=0)
        lon, lat = centroid

        if not (min_lon <= lon <= max_lon and min_lat <= lat <= max_lat):
            continue

        out.append((point_idx, poly_coords))

    return out


m = folium.Map(location=CENTER, zoom_start=ZOOM_START, tiles=None)

# Base map should be in control
folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_nolabels/{z}/{x}/{y}{r}.png",
    attr="&copy; OpenStreetMap contributors &copy; CARTO",
    name="Base map",
    overlay=False,
    control=True
).add_to(m)

folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_only_labels/{z}/{x}/{y}{r}.png",
    attr="&copy; OpenStreetMap contributors &copy; CARTO",
    name="Labels",
    overlay=True,
    control=True
).add_to(m)

# Heatmap as real layer group
heat_fg = folium.FeatureGroup(name="Heatmap", show=False, control=True)
heat_data = [[row["site_lat"], row["site_lon"], row["charger_count"]] for _, row in df_map.iterrows()]
HeatMap(
    heat_data,
    radius=25,
    blur=18,
    min_opacity=0.25,
    max_zoom=12
).add_to(heat_fg)
heat_fg.add_to(m)

# Voronoi
voronoi_fg = folium.FeatureGroup(name="Voronoi", show=False, control=True)
vor_regions = simple_voronoi_regions(df_map, radius=5)

for idx, poly_coords in vor_regions:
    row = df_map.iloc[idx]
    operator = row["OperatorName"]
    color = operator_colors.get(operator, "red")
    coords = [[lat, lon] for lon, lat in poly_coords]

    tooltip = f"{operator} | {int(row['charger_count'])} chargers"
    if "city" in row and pd.notna(row["city"]):
        tooltip += f" | {row['city']}"

    folium.Polygon(
        locations=coords,
        color=color,
        weight=1,
        fill=False,
        opacity=0.5,
        tooltip=tooltip
    ).add_to(voronoi_fg)

voronoi_fg.add_to(m)

# Operator layers
for operator in operators:
    fg = folium.FeatureGroup(
        name=f"{operator} ({station_counts.get(operator, 0)})",
        show=True,
        control=True
    )

    df_op = df_map[df_map["OperatorName"] == operator].copy()
    color = operator_colors[operator]

    for _, row in df_op.iterrows():
        parts = [f"<b>{operator}</b>"]
        parts.append(f"Chargers: {int(row['charger_count'])}")

        if "city" in row and pd.notna(row["city"]):
            parts.append(f"City: {row['city']}")
        if "postal_code" in row and pd.notna(row["postal_code"]):
            parts.append(f"Postal code: {row['postal_code']}")
        if "street" in row and pd.notna(row["street"]):
            addr = str(row["street"])
            if "house_num" in row and pd.notna(row["house_num"]):
                addr += f" {row['house_num']}"
            parts.append(f"Address: {addr}")

        popup_html = "<br>".join(parts)

        folium.CircleMarker(
            location=[row["site_lat"], row["site_lon"]],
            radius=min(4 + row["charger_count"] * 0.35, 16),
            color=color,
            weight=2,
            fill=True,
            fill_color=color,
            fill_opacity=0.85,
            tooltip=f"{operator} | {int(row['charger_count'])} chargers",
            popup=folium.Popup(popup_html, max_width=300)
        ).add_to(fg)

    fg.add_to(m)

legend_html = f"""
<div style="
position: fixed;
bottom: 40px; left: 40px; width: 280px;
max-height: 420px;
overflow-y: auto;
background-color: white;
border: 2px solid grey;
z-index: 9999;
font-size: 12px;
padding: 10px;
">
<b>Operators</b><br>
<div style="margin-bottom:8px;"><b>Total stations: {total_stations}</b></div>
"""

for op in operators:
    count = station_counts.get(op, 0)
    legend_html += f"""
    <div style="margin-top:4px;">
        <span style="
            display:inline-block;
            width:12px;
            height:12px;
            background:{operator_colors[op]};
            margin-right:6px;
            border:1px solid #333;
        "></span>{op} ({count})
    </div>
    """

legend_html += "</div>"
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl(collapsed=False).add_to(m)

m

In [ ]:
m.get_root().header.add_child(folium.Element("""
    <link rel="shortcut icon" href="https://algotecture.net/img/favicon.ico" type="image/x-icon">
    <title>AlgoTecture Map</title>
"""))
m.save("evmap/index.html")

In [ ]:
import pandas as pd
import numpy as np
import folium
from sklearn.cluster import DBSCAN
import requests
from folium.plugins import HeatMap

url = "https://data.geo.admin.ch/ch.bfe.ladestellen-elektromobilitaet/data/ch.bfe.ladestellen-elektromobilitaet.json"
data = requests.get(url).json()

frames = []

for op in data["EVSEData"]:
    operator_name = op.get("OperatorName")
    records = op.get("EVSEDataRecord", [])

    if not records:
        continue

    df_op = pd.json_normalize(records)
    df_op["OperatorName"] = operator_name
    frames.append(df_op)

df_all = pd.concat(frames, ignore_index=True)

print(df_all.shape)
print(df_all.columns.tolist())
# =========================================================
# CONFIG
# =========================================================
CENTER = [47.38922397548196, 8.517438420931438]
ZOOM_START = 10
EPS_METERS = 30  # clustering radius for site detection

# =========================================================
# EXPECTED INPUT
# =========================================================
# df_all = raw flattened EVSE dataframe across all operators
#
# Required columns:
# - OperatorName
# - EvseID
# - ChargingFacilities
# - Address.City
# - Address.PostalCode
# - Address.Street
# - Address.HouseNum
# - GeoCoordinates.Google
#
# Example:
# df_all = pd.concat(list_of_operator_dfs, ignore_index=True)

# =========================================================
# HELPERS
# =========================================================
def parse_google_coords(value):
    if pd.isna(value):
        return pd.Series([np.nan, np.nan])

    s = str(value).strip()
    if not s or "None" in s:
        return pd.Series([np.nan, np.nan])

    parts = s.split()
    if len(parts) != 2:
        return pd.Series([np.nan, np.nan])

    try:
        return pd.Series([float(parts[0]), float(parts[1])])
    except ValueError:
        return pd.Series([np.nan, np.nan])


def extract_max_power(facilities):
    if not isinstance(facilities, list):
        return np.nan

    vals = []
    for f in facilities:
        if isinstance(f, dict):
            p = f.get("power")
            if p is not None:
                try:
                    vals.append(float(p))
                except (TypeError, ValueError):
                    pass

    return max(vals) if vals else np.nan


def extract_powertypes(facilities):
    if not isinstance(facilities, list):
        return None

    vals = []
    for f in facilities:
        if isinstance(f, dict):
            pt = f.get("powertype")
            if pt is not None and str(pt).strip() and str(pt).strip().lower() != "none":
                vals.append(str(pt).strip())

    vals = sorted(set(vals))
    return ", ".join(vals) if vals else None


def combine_text_values(values):
    items = set()
    for v in values:
        if pd.notna(v) and str(v).strip():
            for part in str(v).split(","):
                part = part.strip()
                if part:
                    items.add(part)
    return ", ".join(sorted(items)) if items else None


def format_address(row):
    street = str(row["street"]).strip() if pd.notna(row.get("street")) else ""
    house_num = str(row["house_num"]).strip() if pd.notna(row.get("house_num")) else ""
    postal_code = str(row["postal_code"]).strip() if pd.notna(row.get("postal_code")) else ""
    city = str(row["city"]).strip() if pd.notna(row.get("city")) else ""

    line1 = " ".join(x for x in [street, house_num] if x)
    line2 = " ".join(x for x in [postal_code, city] if x)

    if line1 and line2:
        return f"{line1}, {line2}"
    if line1:
        return line1
    if line2:
        return line2
    return None


def cluster_operator_sites(df_op, eps_meters=30):
    """
    Cluster one operator's EVSE points into site clusters using DBSCAN haversine.
    Returns df_op with cluster_id.
    """
    df_op = df_op.copy()

    if len(df_op) == 0:
        df_op["cluster_id"] = []
        return df_op

    coords_rad = np.radians(df_op[["lat", "lon"]].to_numpy())
    earth_radius_m = 6371000
    eps = eps_meters / earth_radius_m

    db = DBSCAN(eps=eps, min_samples=1, metric="haversine")
    df_op["cluster_id"] = db.fit_predict(coords_rad)

    return df_op


# =========================================================
# PREPARE RAW DATA
# =========================================================
df = df_all.copy()

# parse lat/lon from GeoCoordinates.Google
df[["lat", "lon"]] = df["GeoCoordinates.Google"].apply(parse_google_coords)

df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

# charger count basis
df["EvseID"] = df["EvseID"].astype(str)

# extract power and powertype from ChargingFacilities
df["max_power_kw"] = df["ChargingFacilities"].apply(extract_max_power)
df["powertypes"] = df["ChargingFacilities"].apply(extract_powertypes)

# keep only rows needed for clustering
df = df.dropna(subset=["OperatorName", "lat", "lon"]).reset_index(drop=True)

print("Raw EVSE rows:", len(df))
print("Operators:", df["OperatorName"].nunique())

# =========================================================
# CLUSTER PER OPERATOR
# =========================================================
clustered_parts = []

for operator, df_op in df.groupby("OperatorName", dropna=False):
    df_clustered = cluster_operator_sites(df_op, eps_meters=EPS_METERS)
    df_clustered["site_key"] = df_clustered["OperatorName"].astype(str) + "__" + df_clustered["cluster_id"].astype(str)
    clustered_parts.append(df_clustered)

df_clustered_all = pd.concat(clustered_parts, ignore_index=True)

print("Clustered EVSE rows:", len(df_clustered_all))

# =========================================================
# AGGREGATE TO SITE LEVEL
# =========================================================
agg_dict = {
    "OperatorName": ("OperatorName", "first"),
    "charger_count": ("EvseID", "count"),
    "site_lat": ("lat", "mean"),
    "site_lon": ("lon", "mean"),
    "city": ("Address.City", "first"),
    "postal_code": ("Address.PostalCode", "first"),
    "street": ("Address.Street", "first"),
    "house_num": ("Address.HouseNum", "first"),
    "max_power_kw": ("max_power_kw", "max"),
    "powertypes": ("powertypes", combine_text_values),
}

if "Accessibility" in df_clustered_all.columns:
    agg_dict["Accessibility"] = ("Accessibility", "first")

if "IsOpen24Hours" in df_clustered_all.columns:
    agg_dict["IsOpen24Hours"] = ("IsOpen24Hours", "first")

df_all_sites_clustered = (
    df_clustered_all
    .groupby("site_key", dropna=False)
    .agg(**agg_dict)
    .reset_index(drop=True)
)

print("Mapped sites:", len(df_all_sites_clustered))
print("Operators in mapped sites:", df_all_sites_clustered["OperatorName"].nunique())

# =========================================================
# OPTIONAL: MERGE EXACT DUPLICATE SITE POINTS PER OPERATOR
# =========================================================
df_map = df_all_sites_clustered.copy()

df_map["site_lat"] = pd.to_numeric(df_map["site_lat"], errors="coerce")
df_map["site_lon"] = pd.to_numeric(df_map["site_lon"], errors="coerce")
df_map["charger_count"] = pd.to_numeric(df_map["charger_count"], errors="coerce").fillna(1)
df_map["max_power_kw"] = pd.to_numeric(df_map["max_power_kw"], errors="coerce")

df_map = df_map.dropna(subset=["site_lat", "site_lon", "OperatorName"]).reset_index(drop=True)

df_map = (
    df_map.groupby(["OperatorName", "site_lat", "site_lon"], dropna=False)
    .agg(
        charger_count=("charger_count", "sum"),
        city=("city", "first"),
        postal_code=("postal_code", "first"),
        street=("street", "first"),
        house_num=("house_num", "first"),
        max_power_kw=("max_power_kw", "max"),
        powertypes=("powertypes", combine_text_values),
        Accessibility=("Accessibility", "first") if "Accessibility" in df_map.columns else ("city", "first"),
        IsOpen24Hours=("IsOpen24Hours", "first") if "IsOpen24Hours" in df_map.columns else ("city", "first"),
    )
    .reset_index()
)

# if fallback aggregation above created nonsense because columns were absent, clean it:
if "Accessibility" in df_map.columns and "Accessibility" not in df_all_sites_clustered.columns:
    df_map = df_map.drop(columns=["Accessibility"], errors="ignore")
if "IsOpen24Hours" in df_map.columns and "IsOpen24Hours" not in df_all_sites_clustered.columns:
    df_map = df_map.drop(columns=["IsOpen24Hours"], errors="ignore")

print("Final mapped sites:", len(df_map))

# =========================================================
# OPERATOR COLORS
# =========================================================
palette = [
    "red", "blue", "green", "purple", "orange", "darkred", "lightred",
    "beige", "darkblue", "darkgreen", "cadetblue", "darkpurple",
    "pink", "lightblue", "lightgreen", "gray", "black", "lightgray"
]

operators = sorted(df_map["OperatorName"].dropna().unique())
operator_colors = {op: palette[i % len(palette)] for i, op in enumerate(operators)}

print(operator_colors)

# =========================================================
# BUILD MAP
# =========================================================
m = folium.Map(location=CENTER, zoom_start=ZOOM_START, tiles=None)

folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_nolabels/{z}/{x}/{y}{r}.png",
    attr="&copy; OpenStreetMap contributors &copy; CARTO",
    overlay=False,
    control=False
).add_to(m)

folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_only_labels/{z}/{x}/{y}{r}.png",
    attr="&copy; OpenStreetMap contributors &copy; CARTO",
    overlay=True,
    control=False
).add_to(m)

# ---------------------------------------------------------
# HEATMAP BY MAX POWER
# ---------------------------------------------------------
df_heat = df_map.dropna(subset=["site_lat", "site_lon", "max_power_kw"]).copy()
df_heat["max_power_kw"] = pd.to_numeric(df_heat["max_power_kw"], errors="coerce")
df_heat = df_heat.dropna(subset=["max_power_kw"])

print("Heatmap points:", len(df_heat))
print("Max power range:", df_heat["max_power_kw"].min(), "to", df_heat["max_power_kw"].max())

heat_data_power = [
    [row["site_lat"], row["site_lon"], row["max_power_kw"]]
    for _, row in df_heat.iterrows()
]

heat_fg = folium.FeatureGroup(name="Heatmap: Max power", show=True)

HeatMap(
    heat_data_power,
    radius=28,
    blur=20,
    min_opacity=0.25,
    max_zoom=12
).add_to(heat_fg)

heat_fg.add_to(m)

# =========================================================
# MARKERS BY OPERATOR AS SEPARATE LAYERS
# =========================================================
for operator in operators:
    fg = folium.FeatureGroup(name=operator, show=False)
    df_op = df_map[df_map["OperatorName"] == operator].copy()
    color = operator_colors[operator]

    for _, row in df_op.iterrows():
        address_str = format_address(row)

        parts = [f"<b>{operator}</b>"]
        parts.append(f"Chargers: {int(row['charger_count'])}")

        if address_str:
            parts.append(f"Address: {address_str}")

        if pd.notna(row.get("max_power_kw")):
            parts.append(f"Max power: {int(row['max_power_kw'])} kW")

        if pd.notna(row.get("powertypes")) and str(row.get("powertypes")).strip():
            parts.append(f"Power type: {row['powertypes']}")

        if "Accessibility" in row and pd.notna(row.get("Accessibility")):
            parts.append(f"Accessibility: {row['Accessibility']}")

        if "IsOpen24Hours" in row and pd.notna(row.get("IsOpen24Hours")):
            parts.append(f"24/7: {'Yes' if bool(row['IsOpen24Hours']) else 'No'}")

        popup_html = "<br>".join(parts)

        tooltip_parts = [operator, f"{int(row['charger_count'])} chargers"]
        if pd.notna(row.get("city")):
            tooltip_parts.append(str(row["city"]))

        folium.CircleMarker(
            location=[row["site_lat"], row["site_lon"]],
            radius=min(4 + row["charger_count"] * 0.35, 16),
            color=color,
            weight=2,
            fill=True,
            fill_color=color,
            fill_opacity=0.85,
            tooltip=" | ".join(tooltip_parts),
            popup=folium.Popup(popup_html, max_width=320)
        ).add_to(fg)

    fg.add_to(m)

# =========================================================
# LEGEND
# =========================================================
legend_html = """
<div style="
position: fixed;
bottom: 40px; left: 40px; width: 260px;
max-height: 420px;
overflow-y: auto;
background-color: white;
border: 2px solid grey;
z-index: 9999;
font-size: 12px;
padding: 10px;
">
<b>Operators</b><br>
"""
for op in operators:
    legend_html += f"""
    <div style="margin-top:4px;">
        <span style="
            display:inline-block;
            width:12px;
            height:12px;
            background:{operator_colors[op]};
            margin-right:6px;
            border:1px solid #333;
        "></span>{op}
    </div>
    """
legend_html += "</div>"

m.get_root().html.add_child(folium.Element(legend_html))

# =========================================================
# LAYER CONTROL
# =========================================================
folium.LayerControl(collapsed=True).add_to(m)


m

In [ ]:
import pandas as pd
import numpy as np
import folium
from sklearn.cluster import DBSCAN
import requests
from folium.plugins import HeatMap

# =========================================================
# LOAD DATA
# =========================================================
url = "https://data.geo.admin.ch/ch.bfe.ladestellen-elektromobilitaet/data/ch.bfe.ladestellen-elektromobilitaet.json"
data = requests.get(url).json()

frames = []

for op in data["EVSEData"]:
    operator_name = op.get("OperatorName")
    records = op.get("EVSEDataRecord", [])

    if not records:
        continue

    df_op = pd.json_normalize(records)
    df_op["OperatorName"] = operator_name
    frames.append(df_op)

df_all = pd.concat(frames, ignore_index=True)

print(df_all.shape)
print(df_all.columns.tolist())

# =========================================================
# CONFIG
# =========================================================
CENTER = [47.38922397548196, 8.517438420931438]
ZOOM_START = 10
EPS_METERS = 30  # clustering radius for site detection

# =========================================================
# HELPERS
# =========================================================
def parse_google_coords(value):
    if pd.isna(value):
        return pd.Series([np.nan, np.nan])

    s = str(value).strip()
    if not s or "None" in s:
        return pd.Series([np.nan, np.nan])

    parts = s.split()
    if len(parts) != 2:
        return pd.Series([np.nan, np.nan])

    try:
        return pd.Series([float(parts[0]), float(parts[1])])
    except ValueError:
        return pd.Series([np.nan, np.nan])


def extract_max_power(facilities):
    if not isinstance(facilities, list):
        return np.nan

    vals = []
    for f in facilities:
        if isinstance(f, dict):
            p = f.get("power")
            if p is not None:
                try:
                    vals.append(float(p))
                except (TypeError, ValueError):
                    pass

    return max(vals) if vals else np.nan


def extract_powertypes(facilities):
    if not isinstance(facilities, list):
        return None

    vals = []
    for f in facilities:
        if isinstance(f, dict):
            pt = f.get("powertype")
            if pt is not None and str(pt).strip() and str(pt).strip().lower() != "none":
                vals.append(str(pt).strip())

    vals = sorted(set(vals))
    return ", ".join(vals) if vals else None


def combine_text_values(values):
    items = set()
    for v in values:
        if pd.notna(v) and str(v).strip():
            for part in str(v).split(","):
                part = part.strip()
                if part:
                    items.add(part)
    return ", ".join(sorted(items)) if items else None


def format_address(row):
    street = str(row["street"]).strip() if pd.notna(row.get("street")) else ""
    house_num = str(row["house_num"]).strip() if pd.notna(row.get("house_num")) else ""
    postal_code = str(row["postal_code"]).strip() if pd.notna(row.get("postal_code")) else ""
    city = str(row["city"]).strip() if pd.notna(row.get("city")) else ""

    line1 = " ".join(x for x in [street, house_num] if x)
    line2 = " ".join(x for x in [postal_code, city] if x)

    if line1 and line2:
        return f"{line1}, {line2}"
    if line1:
        return line1
    if line2:
        return line2
    return None


def cluster_operator_sites(df_op, eps_meters=30):
    df_op = df_op.copy()

    if len(df_op) == 0:
        df_op["cluster_id"] = []
        return df_op

    coords_rad = np.radians(df_op[["lat", "lon"]].to_numpy())
    earth_radius_m = 6371000
    eps = eps_meters / earth_radius_m

    db = DBSCAN(eps=eps, min_samples=1, metric="haversine")
    df_op["cluster_id"] = db.fit_predict(coords_rad)

    return df_op


# =========================================================
# PREPARE RAW DATA
# =========================================================
df = df_all.copy()

df[["lat", "lon"]] = df["GeoCoordinates.Google"].apply(parse_google_coords)
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
df["EvseID"] = df["EvseID"].astype(str)

df["max_power_kw"] = df["ChargingFacilities"].apply(extract_max_power)
df["powertypes"] = df["ChargingFacilities"].apply(extract_powertypes)

df = df.dropna(subset=["OperatorName", "lat", "lon"]).reset_index(drop=True)

print("Raw EVSE rows:", len(df))
print("Operators:", df["OperatorName"].nunique())

# =========================================================
# CLUSTER PER OPERATOR
# =========================================================
clustered_parts = []

for operator, df_op in df.groupby("OperatorName", dropna=False):
    df_clustered = cluster_operator_sites(df_op, eps_meters=EPS_METERS)
    df_clustered["site_key"] = (
        df_clustered["OperatorName"].astype(str) + "__" + df_clustered["cluster_id"].astype(str)
    )
    clustered_parts.append(df_clustered)

df_clustered_all = pd.concat(clustered_parts, ignore_index=True)

print("Clustered EVSE rows:", len(df_clustered_all))

# =========================================================
# AGGREGATE TO SITE LEVEL
# =========================================================
agg_dict = {
    "OperatorName": ("OperatorName", "first"),
    "charger_count": ("EvseID", "count"),
    "site_lat": ("lat", "mean"),
    "site_lon": ("lon", "mean"),
    "city": ("Address.City", "first"),
    "postal_code": ("Address.PostalCode", "first"),
    "street": ("Address.Street", "first"),
    "house_num": ("Address.HouseNum", "first"),
    "max_power_kw": ("max_power_kw", "max"),
    "powertypes": ("powertypes", combine_text_values),
}

if "Accessibility" in df_clustered_all.columns:
    agg_dict["Accessibility"] = ("Accessibility", "first")

if "IsOpen24Hours" in df_clustered_all.columns:
    agg_dict["IsOpen24Hours"] = ("IsOpen24Hours", "first")

df_all_sites_clustered = (
    df_clustered_all
    .groupby("site_key", dropna=False)
    .agg(**agg_dict)
    .reset_index(drop=True)
)

print("Mapped sites:", len(df_all_sites_clustered))
print("Operators in mapped sites:", df_all_sites_clustered["OperatorName"].nunique())

# =========================================================
# MERGE EXACT DUPLICATE SITE POINTS PER OPERATOR
# =========================================================
df_map = df_all_sites_clustered.copy()

df_map["site_lat"] = pd.to_numeric(df_map["site_lat"], errors="coerce")
df_map["site_lon"] = pd.to_numeric(df_map["site_lon"], errors="coerce")
df_map["charger_count"] = pd.to_numeric(df_map["charger_count"], errors="coerce").fillna(1)
df_map["max_power_kw"] = pd.to_numeric(df_map["max_power_kw"], errors="coerce")

df_map = df_map.dropna(subset=["site_lat", "site_lon", "OperatorName"]).reset_index(drop=True)

agg_dict_2 = {
    "charger_count": ("charger_count", "sum"),
    "city": ("city", "first"),
    "postal_code": ("postal_code", "first"),
    "street": ("street", "first"),
    "house_num": ("house_num", "first"),
    "max_power_kw": ("max_power_kw", "max"),
    "powertypes": ("powertypes", combine_text_values),
}

if "Accessibility" in df_map.columns:
    agg_dict_2["Accessibility"] = ("Accessibility", "first")

if "IsOpen24Hours" in df_map.columns:
    agg_dict_2["IsOpen24Hours"] = ("IsOpen24Hours", "first")

df_map = (
    df_map.groupby(["OperatorName", "site_lat", "site_lon"], dropna=False)
    .agg(**agg_dict_2)
    .reset_index()
)

print("Final mapped sites:", len(df_map))

# =========================================================
# POWER BANDS
# =========================================================
power_bins = [
    (0, 20, "0-20 kW", "#9e9e9e"),
    (20, 50, "20-50 kW", "#66bb6a"),
    (50, 100, "50-100 kW", "#42a5f5"),
    (100, 200, "100-200 kW", "#7e57c2"),
    (200, 300, "200-300 kW", "#ef5350"),
    (300, np.inf, ">300 kW", "#000000"),
]

def classify_power_band(power):
    if pd.isna(power):
        return "Unknown"
    for low, high, label, color in power_bins:
        if low <= power < high:
            return label
    return "Unknown"

def power_band_color(label):
    for _, _, band_label, color in power_bins:
        if band_label == label:
            return color
    return "#bdbdbd"

df_map["power_band"] = df_map["max_power_kw"].apply(classify_power_band)

print(df_map["power_band"].value_counts(dropna=False))

# =========================================================
# BUILD MAP
# =========================================================
m = folium.Map(location=CENTER, zoom_start=ZOOM_START, tiles=None)

folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_nolabels/{z}/{x}/{y}{r}.png",
    attr="&copy; OpenStreetMap contributors &copy; CARTO",
    overlay=False,
    control=False
).add_to(m)

folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_only_labels/{z}/{x}/{y}{r}.png",
    attr="&copy; OpenStreetMap contributors &copy; CARTO",
    overlay=True,
    control=False
).add_to(m)

# =========================================================
# HEATMAP BY MAX POWER
# =========================================================
df_heat = df_map.dropna(subset=["site_lat", "site_lon", "max_power_kw"]).copy()
df_heat["max_power_kw"] = pd.to_numeric(df_heat["max_power_kw"], errors="coerce")
df_heat = df_heat.dropna(subset=["max_power_kw"])

print("Heatmap points:", len(df_heat))
print("Max power range:", df_heat["max_power_kw"].min(), "to", df_heat["max_power_kw"].max())

heat_data_power = [
    [row["site_lat"], row["site_lon"], row["max_power_kw"]]
    for _, row in df_heat.iterrows()
]

heat_fg = folium.FeatureGroup(name="Heatmap: Max power", show=True)

HeatMap(
    heat_data_power,
    radius=28,
    blur=20,
    min_opacity=0.25,
    max_zoom=12
).add_to(heat_fg)

heat_fg.add_to(m)

# =========================================================
# MARKERS BY POWER BAND
# =========================================================
band_order = [label for _, _, label, _ in power_bins]

for band_label in band_order:
    df_band = df_map[df_map["power_band"] == band_label].copy()
    band_site_count = len(df_band)
    band_charger_count = int(df_band["charger_count"].sum()) if len(df_band) else 0

    # choose what you want to show in layer control:
    # sites = number of clustered locations
    # chargers = total chargers in that band
    layer_name = f"{band_label} ({band_charger_count})"
    # alternative:
    # layer_name = f"{band_label} ({band_site_count} sites, {band_charger_count} chargers)"

    fg = folium.FeatureGroup(name=layer_name, show=False)
    color = power_band_color(band_label)

    for _, row in df_band.iterrows():
        address_str = format_address(row)

        parts = [f"<b>{row['OperatorName']}</b>"]
        parts.append(f"Chargers: {int(row['charger_count'])}")

        if address_str:
            parts.append(f"Address: {address_str}")

        if pd.notna(row.get("max_power_kw")):
            parts.append(f"Max power: {int(row['max_power_kw'])} kW")

        if pd.notna(row.get("powertypes")) and str(row.get("powertypes")).strip():
            parts.append(f"Power type: {row['powertypes']}")

        if "Accessibility" in row and pd.notna(row.get("Accessibility")):
            parts.append(f"Accessibility: {row['Accessibility']}")

        if "IsOpen24Hours" in row and pd.notna(row.get("IsOpen24Hours")):
            parts.append(f"24/7: {'Yes' if bool(row['IsOpen24Hours']) else 'No'}")

        #parts.append(f"Power band: {band_label}")

        popup_html = "<br>".join(parts)

        tooltip_parts = [band_label, f"{int(row['charger_count'])} chargers"]
        if pd.notna(row.get("city")):
            tooltip_parts.append(str(row["city"]))

        folium.CircleMarker(
            location=[row["site_lat"], row["site_lon"]],
            radius=min(4 + row["charger_count"] * 0.35, 16),
            color=color,
            weight=2,
            fill=True,
            fill_color=color,
            fill_opacity=0.85,
            tooltip=" | ".join(tooltip_parts),
            popup=folium.Popup(popup_html, max_width=320)
        ).add_to(fg)

    fg.add_to(m)

# =========================================================
# OPTIONAL UNKNOWN POWER LAYER
# =========================================================
if (df_map["power_band"] == "Unknown").any():
    df_unknown = df_map[df_map["power_band"] == "Unknown"].copy()
    unknown_site_count = len(df_unknown)
    unknown_charger_count = int(df_unknown["charger_count"].sum()) if len(df_unknown) else 0
    
    fg = folium.FeatureGroup(
        name=f"Unknown power ({unknown_charger_count})",
        show=False
    )    
    for _, row in df_map[df_map["power_band"] == "Unknown"].iterrows():
        address_str = format_address(row)

        parts = [f"<b>{row['OperatorName']}</b>"]
        parts.append(f"Chargers: {int(row['charger_count'])}")
        if address_str:
            parts.append(f"Address: {address_str}")
        parts.append("Max power: n/a")

        popup_html = "<br>".join(parts)

        folium.CircleMarker(
            location=[row["site_lat"], row["site_lon"]],
            radius=min(4 + row["charger_count"] * 0.35, 16),
            color="#bdbdbd",
            weight=2,
            fill=True,
            fill_color="#bdbdbd",
            fill_opacity=0.75,
            tooltip="Unknown power",
            popup=folium.Popup(popup_html, max_width=320)
        ).add_to(fg)

    fg.add_to(m)

# =========================================================
# LEGEND
# =========================================================
legend_html = """
<div style="
position: fixed;
bottom: 40px; left: 40px; width: 220px;
max-height: 320px;
overflow-y: auto;
background-color: white;
border: 2px solid grey;
z-index: 9999;
font-size: 12px;
padding: 10px;
">
<b>Max power bands</b><br>
"""

for _, _, label, color in power_bins:
    legend_html += f"""
    <div style="margin-top:4px;">
        <span style="
            display:inline-block;
            width:12px;
            height:12px;
            background:{color};
            margin-right:6px;
            border:1px solid #333;
        "></span>{label}
    </div>
    """

legend_html += "</div>"

m.get_root().html.add_child(folium.Element(legend_html))

# =========================================================
# LAYER CONTROL
# =========================================================
folium.LayerControl(collapsed=True).add_to(m)

m


In [ ]:

m.get_root().header.add_child(folium.Element("""
    <link rel="shortcut icon" href="https://algotecture.net/img/favicon.ico" type="image/x-icon">
    <title>AlgoTecture Map</title>
"""))
m.save("map/index.html")